In [0]:
from pyspark.sql import functions as F

nombres = {1: "Montevideo", 2: "Artigas", 3: "Canelones", 4: "Cerro Largo", 5: "Colonia",
           6: "Durazno", 7: "Flores", 8: "Florida", 9: "Lavalleja", 10: "Maldonado",
           11: "Paysandú", 12: "Río Negro", 13: "Rivera", 14: "Rocha", 15: "Salto",
           16: "San José", 17: "Soriano", 18: "Tacuarembó", 19: "Treinta y Tres"}
mapa = F.create_map([F.lit(x) for kv in nombres.items() for x in kv])

cols = ["ID", "nper", "anio", "mes", "dpto",
        "e26", "e27", "pobre17", "indig17", "W_ANO"]

base = (spark.table("workspace.pobreza.ech_2024_bronze").select(cols)
        .unionByName(spark.table("workspace.pobreza.ech_2025_bronze").select(cols))
        .withColumn("nom_dpto", mapa[F.col("dpto")]))

(base.write.mode("overwrite").option("overwriteSchema", True)
     .saveAsTable("workspace.pobreza.personas_silver"))

display(base.select("dpto", "nom_dpto").distinct().orderBy("dpto"))